In [105]:
# Import Required Libraries
import pandas as pd
import numpy as np
import altair as alt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

# Enable Altair's data transformer for large datasets
alt.data_transformers.enable('default', max_rows=None)
alt.renderers.enable('default')

RendererRegistry.enable('default')

In [106]:
# load the clustering data for feature selection and clustering steps
selected_features_minus_noise = pd.read_csv('../../data/clustering/selected_features_minus_noise.csv')
pre_covid_sf_minus_noise = pd.read_csv('../../data/clustering/pre_covid_sf_minus_noise.csv')
covid_sf_minus_noise = pd.read_csv('../../data/clustering/covid_sf_minus_noise.csv')
post_covid_sf_minus_noise = pd.read_csv('../../data/clustering/post_covid_sf_minus_noise.csv')

feature_cols = [col for col in selected_features_minus_noise.columns if col not in ['Ticker', 'Company', 'Industrykey', 'Category']]
reduced_X_scaled_minus_noise = selected_features_minus_noise[feature_cols].values

selected_features_minus_noise

,Ticker,Company,Industrykey,Category,Avg_EPS_Change_4Q_mean,Avg_NIL_Change_4Q_mean,NetWorth_Change_mean,NetWorth_Volatility_2Q_max,Volume_accum_max,MarketCap_max,MarketCap_min,Volume_Change_std,EPS_Volatility_2Q_max,Return_mean,Volatility_2Q_max,Return_skew
0,A,Agilent Technologies,diagnostics-research,sp500,-0.076244,4.114504,-0.405524,-0.225121,-0.212947,0.005373,0.239710,-0.530947,0.031854,0.091116,-0.483580,0.096013
1,ABBV,AbbVie,drug-manufacturers-general,sp500,0.471927,0.455383,0.804841,0.141936,1.103978,5.189856,3.091809,0.113301,0.243750,0.501265,-0.340968,0.625316
2,ABNB,Airbnb,travel-services,sp500,10.113777,7.142249,1.399025,-0.003276,0.396645,1.025774,0.895582,-1.021104,1.286021,-1.224523,0.443061,-0.228789
3,ABT,Abbott Laboratories,medical-devices,sp500,0.484913,0.398751,0.157517,0.212518,0.852305,3.025666,2.361383,0.588628,0.068567,-0.120219,-0.632658,0.119561
4,ACGL,Arch Capital Group,insurance-diversified,sp500,0.213532,0.210093,0.374298,0.012921,-0.229294,-0.103846,-0.362123,0.492001,-0.222774,0.062081,-0.544019,-0.042160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
477,XYZ,"Block, Inc.",software-infrastructure,sp500,1.108501,1.403034,3.952880,1.827585,1.757268,0.930182,-0.419073,1.896908,0.628494,2.140524,2.206237,0.889641
478,YUM,Yum! Brands,restaurants,sp500,0.101693,0.064127,-4.588531,3.138669,-0.010206,0.025074,0.620430,-0.281343,-0.202228,-0.228233,0.202196,-0.663321
479,ZBH,Zimmer Biomet,medical-devices,sp500,1.996738,1.841417,-0.033517,-0.060907,-0.255237,-0.184204,0.380765,0.184212,3.063811,-1.115613,-0.094686,-0.645380
480,ZBRA,Zebra Technologies,communication-equipment,sp500,-0.684667,-0.402207,0.426374,-0.026450,-0.436228,-0.261039,-0.325252,0.079199,0.347151,0.295716,0.494563,0.395540


In [107]:
def compute_kmeans_metrics(X, k_range=range(2, 10)):
    """Run KMeans for each k and return metric lists + numpy arrays."""
    inertias, silhouette_scores, davies_bouldin_scores, calinski_harabasz_scores = [], [], [], []
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        inertias.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(X, labels))
        davies_bouldin_scores.append(davies_bouldin_score(X, labels))
        calinski_harabasz_scores.append(calinski_harabasz_score(X, labels))
    return (
        inertias, silhouette_scores, davies_bouldin_scores, calinski_harabasz_scores,
        np.array(inertias), np.array(silhouette_scores),
        np.array(davies_bouldin_scores), np.array(calinski_harabasz_scores)
    )


def build_metric_dataframes(k_range, inertias, silhouette_scores, davies_bouldin_scores, calinski_harabasz_scores):
    """Build inertia and shared-metrics DataFrames for charting."""
    k_list = list(k_range)
    inertia_df = pd.DataFrame({'k': k_list, 'Inertia': inertias})
    shared_metrics_df = pd.DataFrame([
        {'k': k, 'Method': 'K-Means', 'Silhouette': s, 'Davies-Bouldin': d, 'Calinski-Harabasz': c}
        for k, s, d, c in zip(k_list, silhouette_scores, davies_bouldin_scores, calinski_harabasz_scores)
    ])
    return inertia_df, shared_metrics_df


def plot_kmeans_metrics(inertia_df, shared_metrics_df):
    """Return a 2×2 Altair grid of KMeans evaluation metric charts."""
    method_color = alt.Color(
        'Method:N',
        scale=alt.Scale(domain=['K-Means'], range=['#4C78A8']),
        title='Method', legend=None
    )
    base_km     = alt.Chart(inertia_df).encode(
        x=alt.X('k:O', title='Number of Clusters (k)', axis=alt.Axis(labelAngle=0))
    )
    base_shared = alt.Chart(shared_metrics_df).encode(
        x=alt.X('k:O', title='Number of Clusters (k / n)', axis=alt.Axis(labelAngle=0)),
        color=method_color
    )
    elbow = base_km.mark_line(point=True, color='#4C78A8').encode(
        y=alt.Y('Inertia:Q', title='Inertia', scale=alt.Scale(zero=False)),
        tooltip=['k', 'Inertia']
    ).properties(title='Elbow Method / Inertia  (K-Means Only)', width=alt.Step(60), height=300)

    silhouette_chart = base_shared.mark_line(point=True).encode(
        y=alt.Y('Silhouette:Q', title='Silhouette Score', scale=alt.Scale(zero=False)),
        tooltip=['k', 'Method', 'Silhouette']
    ).properties(title='Silhouette Score (Higher is Better)', width=alt.Step(60), height=300)

    davies_chart = base_shared.mark_line(point=True).encode(
        y=alt.Y('Davies-Bouldin:Q', title='Davies-Bouldin Index', scale=alt.Scale(zero=False)),
        tooltip=['k', 'Method', 'Davies-Bouldin']
    ).properties(title='Davies-Bouldin Index (Lower is Better)', width=alt.Step(60), height=300)

    calinski_chart = base_shared.mark_line(point=True).encode(
        y=alt.Y('Calinski-Harabasz:Q', title='Calinski-Harabasz Score', scale=alt.Scale(zero=False)),
        tooltip=['k', 'Method', 'Calinski-Harabasz']
    ).properties(title='Calinski-Harabasz Score (Higher is Better)', width=alt.Step(60), height=300)

    return (
        (elbow | silhouette_chart) & (davies_chart | calinski_chart)
    ).resolve_scale(y='independent').configure_axis(
        labelFontSize=14,
        titleFontSize=14,
    ).configure_title(fontSize=18)


k_range = range(2, 10)

(inertias, silhouette_scores, davies_bouldin_scores, calinski_harabasz_scores,
 inertias_arr, silhouette_arr, db_arr, ch_arr) = compute_kmeans_metrics(reduced_X_scaled_minus_noise, k_range)

inertia_df, shared_metrics_df = build_metric_dataframes(
    k_range, inertias, silhouette_scores, davies_bouldin_scores, calinski_harabasz_scores
)

plot_kmeans_metrics(inertia_df, shared_metrics_df)


alt.VConcatChart(...)

In [108]:

def minmax(x):
    x = np.array(x)
    return (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x))


def compute_combined_score(inertias, silhouette_scores, db_scores, ch_scores):
    """Normalize and combine four K-Means metrics into a single score (higher is better)."""
    norm_inertia    = 1 - minmax(inertias)      # lower is better → invert
    norm_silhouette = minmax(silhouette_scores)
    norm_db         = 1 - minmax(db_scores)     # lower is better → invert
    norm_ch         = minmax(ch_scores)
    return (norm_inertia + norm_silhouette + norm_db + norm_ch) / 4


def build_metrics_df(k_range, inertias, silhouette_scores, db_scores, ch_scores, combined_score):
    """Build a summary DataFrame of all K-Means evaluation metrics."""
    return pd.DataFrame({
        'k': list(k_range),
        'Inertia': inertias,
        'Silhouette': silhouette_scores,
        'Davies-Bouldin': db_scores,
        'Calinski-Harabasz': ch_scores,
        'Combined_Score': combined_score
    })


def select_optimal_k(k_range, combined_score, top_n=4):
    """Return the top-n k values by combined score and the single best k."""
    top_idx = np.argsort(combined_score)[-top_n:][::-1]
    top_k   = [list(k_range)[i] for i in top_idx]
    return top_k, top_k[0]


def find_optimal_k(X_data, k_range=range(2, 10)):
    """Find optimal k for a dataset using combined normalized metrics."""
    inertias_era, silhouette_era, db_era, ch_era = [], [], [], []
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X_data)
        inertias_era.append(kmeans.inertia_)
        silhouette_era.append(silhouette_score(X_data, labels))
        db_era.append(davies_bouldin_score(X_data, labels))
        ch_era.append(calinski_harabasz_score(X_data, labels))
    combined_era = compute_combined_score(inertias_era, silhouette_era, db_era, ch_era)
    return list(k_range)[np.argmax(combined_era)]


def apply_kmeans(X, n_clusters):
    """Fit KMeans and return (model, labels)."""
    model = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    labels = model.fit_predict(X)
    return model, labels


def plot_combined_score(k_range, combined_score):
    """Return an Altair chart of the combined normalized score vs k."""
    df = pd.DataFrame([
        {'k': k, 'Method': 'K-Means', 'Combined_Score': s}
        for k, s in zip(list(k_range), combined_score)
    ])
    return (
        alt.Chart(df).mark_line(point=True).encode(
            x=alt.X('k:O', title='Number of Clusters (k / n)', axis=alt.Axis(labelAngle=0)),
            y=alt.Y('Combined_Score:Q', title='Combined Score (Normalized)', scale=alt.Scale(zero=False)),
            color=alt.Color(
                'Method:N',
                scale=alt.Scale(domain=['K-Means'], range=['#4C78A8']),
                title='Method'
            ),
            tooltip=['k', 'Method', alt.Tooltip('Combined_Score:Q', format='.4f')]
        ).properties(
            title='Combined Normalized Score — K-Means (Higher is Better)',
            width=alt.Step(60),
            height=320
        ).configure_axis(
            labelFontSize=14,
            titleFontSize=14,
        ).configure_title(
            fontSize=18
        )
    )

combined_score = compute_combined_score(inertias_arr, silhouette_arr, db_arr, ch_arr)
metrics_df     = build_metrics_df(k_range, inertias, silhouette_scores, davies_bouldin_scores,
                                   calinski_harabasz_scores, combined_score)
top_k, optimal_k = select_optimal_k(k_range, combined_score)

_, kmeans_labels = apply_kmeans(reduced_X_scaled_minus_noise, optimal_k)
selected_features_minus_noise['KMeans_Cluster'] = kmeans_labels

print("K-Means Metrics Summary:")
print(metrics_df.to_string(index=False))
print(f"\nTop 4 k by combined normalized metrics (K-Means): {top_k}")
print(f"Optimal k selected for K-Means: {optimal_k}")

# Calculate optimal k for each era
era_datasets_raw = {
    'All Periods':           selected_features_minus_noise,
    'Pre-COVID (2016-2019)': pre_covid_sf_minus_noise,
    'COVID (2020-2021)':     covid_sf_minus_noise,
    'Post-COVID (2022-2025)': post_covid_sf_minus_noise
}

era_optimal_k = {}
print("\n" + "="*70)
print("Finding optimal k for each era:")
for era_name, era_df in era_datasets_raw.items():
    era_feature_cols = [col for col in era_df.columns if col not in ['Ticker', 'Company', 'Industrykey', 'Category']]
    era_optimal_k[era_name] = find_optimal_k(era_df[era_feature_cols].values)
    print(f"  {era_name}: k = {era_optimal_k[era_name]}")
print("="*70)

plot_combined_score(k_range, combined_score)


K-Means Metrics Summary:
 k      Inertia  Silhouette  Davies-Bouldin  Calinski-Harabasz  Combined_Score
 2 41093.190113    0.796726        0.771434          75.381953        0.500000
 3 36069.575071    0.595979        1.328519          76.207482        0.176062
 4 32117.134849    0.596808        0.998536          76.546184        0.369381
 5 28342.829744    0.604704        1.253735          80.798621        0.338437
 6 25452.532270    0.573727        0.983922          82.638725        0.485909
 7 22188.217261    0.488502        0.956889          90.478048        0.546478
 8 19129.605218    0.533525        0.785000         100.589807        0.777640
 9 17345.179740    0.423060        0.934288         102.948769        0.676917

Top 4 k by combined normalized metrics (K-Means): [8, 9, 7, 2]
Optimal k selected for K-Means: 8

Finding optimal k for each era:
  All Periods: k = 8
  Pre-COVID (2016-2019): k = 5
  COVID (2020-2021): k = 4
  Post-COVID (2022-2025): k = 7


alt.Chart(...)

In [109]:
# Function to plot PCA explained variance
def plot_pca_explained_variance(X):
    pca_full = PCA(random_state=42)
    pca_full.fit(X)
    cumsum_variance = np.cumsum(pca_full.explained_variance_ratio_)
    explained_df = pd.DataFrame({
        'Component': np.arange(1, len(cumsum_variance) + 1),
        'CumulativeVariance': cumsum_variance
    })

    return alt.Chart(explained_df).mark_line(point=True).encode(
        x=alt.X('Component:O', title='Number of Components'),
        y=alt.Y('CumulativeVariance:Q', title='Cumulative Explained Variance', scale=alt.Scale(zero=False)),
        tooltip=['Component', alt.Tooltip('CumulativeVariance:Q', format='.2%')]
        ).properties(
            title='PCA Cumulative Explained Variance',
            width=alt.Step(40),
            height=300
        ).configure_axis(
        labelFontSize=14,
        titleFontSize=14,
        labelAngle=0
        ).configure_title(
            fontSize=18
        )
        
plot_pca_explained_variance(reduced_X_scaled_minus_noise)

alt.Chart(...)

In [110]:
print(selected_features_minus_noise[feature_cols].describe().loc[['mean', 'std']])

      Avg_EPS_Change_4Q_mean  Avg_NIL_Change_4Q_mean  NetWorth_Change_mean  \
mean               -0.129518                0.024111              0.181717   
std                 3.284201                5.039717              4.244176   

      NetWorth_Volatility_2Q_max  Volume_accum_max  MarketCap_max  \
mean                    1.021589          0.708046       0.800491   
std                     3.884484          2.033481       3.214770   

      MarketCap_min  Volume_Change_std  EPS_Volatility_2Q_max  Return_mean  \
mean        0.65472           0.392204               0.735099     0.135371   
std         1.83227           1.882754               2.176994     1.174473   

      Volatility_2Q_max  Return_skew  
mean           0.218435     0.131875  
std            1.054646     0.979786  


In [111]:

HIGHLIGHT_CATS  = {'gig-work', 'mlm-work'}
_CAT_COLOR_MAP  = {'gig-work': '#4C4441', 'mlm-work': '#9C755F'}
PALETTE = [
    '#4C78A8', '#F58518', '#54A24B', '#E45756',
    '#22B6B2', '#FFB5B8', '#9E7BB5', '#B0B0B0',
    '#FFEF6B', '#FF1DA7',
]
META_COLS = ['Ticker', 'Company', 'Industrykey', 'Category']


def build_era_pca_df(era_datasets_raw, era_optimal_k, pca_vis, X_pca_all, feature_cols):
    """Project each era's data into PCA space, run KMeans, and return a combined DataFrame."""
    rows = []
    for era_name, era_df in era_datasets_raw.items():
        era_feature_cols = [c for c in era_df.columns if c not in META_COLS]
        era_X = era_df[era_feature_cols].values
        k_for_era = era_optimal_k[era_name]

        era_labels = KMeans(n_clusters=k_for_era, random_state=42, n_init=20).fit_predict(era_X)
        era_X_pca  = X_pca_all if era_name == 'All Periods' else pca_vis.transform(era_X)

        era_rows = pd.DataFrame(era_X_pca, columns=['PC1', 'PC2'])
        era_rows['Ticker']       = era_df['Ticker'].values
        era_rows['Company']      = era_df['Company'].values
        era_rows['Category']     = era_df['Category'].values
        era_rows['Industrykey']  = era_df['Industrykey'].values
        era_rows['Cluster']      = era_labels.astype(str)
        era_rows['ClusterLabel'] = 'Cluster ' + era_rows['Cluster']
        era_rows['Era']          = era_name
        era_rows['OptimalK']     = k_for_era
        for col in feature_cols[:3]:
            if col in era_df.columns:
                era_rows[col] = era_df[col].values
        rows.append(era_rows)

    return pd.concat(rows, ignore_index=True)


def build_centroids_df(combined_pca_df):
    """Compute per-era cluster centroid coordinates."""
    centroid_rows = []
    for era_name in combined_pca_df['Era'].unique():
        era_data = combined_pca_df[combined_pca_df['Era'] == era_name]
        for cid in sorted(era_data['Cluster'].unique()):
            cluster_data = era_data[era_data['Cluster'] == cid]
            centroid_rows.append({
                'PC1': cluster_data['PC1'].mean(),
                'PC2': cluster_data['PC2'].mean(),
                'Cluster': cid,
                'ClusterLabel': f'Cluster {cid}',
                'Era': era_name
            })
    return pd.DataFrame(centroid_rows)


def build_highlight_df(combined_pca_df, ref_df):
    """Return rows for highlighted tickers (gig-work & mlm-work) with their display color."""
    ticker_color_map = (
        ref_df[ref_df['Category'].isin(HIGHLIGHT_CATS)]
        .set_index('Ticker')['Category']
        .map(_CAT_COLOR_MAP)
        .to_dict()
    )
    highlight = combined_pca_df[combined_pca_df['Ticker'].isin(ticker_color_map)].copy()
    highlight['HighlightColor'] = highlight['Ticker'].map(ticker_color_map)
    return highlight


def plot_clustering_vis(combined_pca_df, centroids_vis_df, highlight_rows, era_optimal_k, pca_vis, feature_cols):
    """Return the interactive Altair PCA scatter chart with era toggle and cluster selection."""
    cluster_scale = alt.Scale(range=PALETTE)
    cluster_sel   = alt.selection_point(fields=['Cluster'])
    era_param     = alt.param(
        name='era_selection', value='All Periods',
        bind=alt.binding_radio(
            options=['All Periods', 'Pre-COVID (2016-2019)', 'COVID (2020-2021)', 'Post-COVID (2022-2025)'],
            name='Time Period: '
        )
    )
    era_filter = alt.datum.Era == era_param

    halo = alt.Chart(combined_pca_df).mark_circle(opacity=0.08).encode(
        x='PC1:Q', y='PC2:Q',
        color=alt.Color('Cluster:N', scale=cluster_scale, legend=None),
        size=alt.condition(cluster_sel, alt.value(2800), alt.value(800))
    ).transform_filter(era_filter)

    scatter_vis = alt.Chart(combined_pca_df).mark_circle(stroke='white', strokeWidth=1.2).encode(
        x=alt.X('PC1:Q',
                title=f'PC1  –  {pca_vis.explained_variance_ratio_[0]:.1%} variance explained',
                axis=alt.Axis(grid=True, gridColor='#EBEBEB', tickCount=6)),
        y=alt.Y('PC2:Q',
                title=f'PC2  –  {pca_vis.explained_variance_ratio_[1]:.1%} variance explained',
                axis=alt.Axis(grid=True, gridColor='#EBEBEB', tickCount=6)),
        color=alt.Color('Cluster:N', scale=cluster_scale, title='Cluster'),
        size=alt.condition(cluster_sel, alt.value(280), alt.value(120)),
        opacity=alt.condition(cluster_sel, alt.value(0.92), alt.value(0.28)),
        tooltip=[
            alt.Tooltip('Company:N',      title='Company'),
            alt.Tooltip('Category:N',     title='Category'),
            alt.Tooltip('ClusterLabel:N', title='Cluster'),
            alt.Tooltip(feature_cols[0] + ':Q', title=feature_cols[0], format='.4f'),
            alt.Tooltip(feature_cols[1] + ':Q', title=feature_cols[1], format='.4f'),
            alt.Tooltip(feature_cols[2] + ':Q', title=feature_cols[2], format='.4f'),
        ]
    ).transform_filter(era_filter)

    text_vis = alt.Chart(combined_pca_df).mark_text(
        fontSize=9, fontWeight=300, dy=-12, color='#444444'
    ).encode(
        x='PC1:Q', y='PC2:Q',
        text=alt.Text('Ticker:N'),
        opacity=alt.condition(cluster_sel, alt.value(0.85), alt.value(0.0))
    ).transform_filter(era_filter)

    centroid_outer = alt.Chart(centroids_vis_df).mark_point(
        shape='diamond', filled=True, size=480, opacity=0.25
    ).encode(
        x='PC1:Q', y='PC2:Q',
        color=alt.Color('Cluster:N', scale=cluster_scale, legend=None)
    ).transform_filter(era_filter)

    centroid_inner = alt.Chart(centroids_vis_df).mark_point(
        shape='diamond', filled=True, size=180, stroke='white', strokeWidth=2
    ).encode(
        x='PC1:Q', y='PC2:Q',
        color=alt.Color('Cluster:N', scale=cluster_scale, legend=None),
        tooltip=[alt.Tooltip('ClusterLabel:N', title='Centroid')]
    ).transform_filter(era_filter)

    highlight_glow = alt.Chart(highlight_rows).mark_circle(opacity=0.22, strokeWidth=0).encode(
        x='PC1:Q', y='PC2:Q',
        color=alt.Color('HighlightColor:N', scale=None, legend=None),
        size=alt.value(1400)
    ).transform_filter(era_filter)

    highlight_pts = alt.Chart(highlight_rows).mark_circle(strokeWidth=2.8, filled=True).encode(
        x='PC1:Q', y='PC2:Q',
        color=alt.Color('HighlightColor:N', scale=None, legend=None),
        stroke=alt.value('white'),
        size=alt.value(310),
        opacity=alt.value(0.95),
        tooltip=[
            alt.Tooltip('Company:N',      title='Company'),
            alt.Tooltip('Category:N',     title='Category'),
            alt.Tooltip('ClusterLabel:N', title='Cluster'),
        ]
    ).transform_filter(era_filter)

    highlight_text = alt.Chart(highlight_rows).mark_text(
        fontSize=11, fontWeight='bold', dy=-14
    ).encode(
        x='PC1:Q', y='PC2:Q',
        text=alt.Text('Ticker:N'),
        color=alt.Color('HighlightColor:N', scale=None, legend=None),
        opacity=alt.value(1.0)
    ).transform_filter(era_filter)

    k_values_text = ' | '.join([f"{era.split(' (')[0]}: k={k}" for era, k in era_optimal_k.items()])

    return (
        halo + scatter_vis + text_vis + centroid_outer + centroid_inner
        + highlight_glow + highlight_pts + highlight_text
    ).add_params(cluster_sel, era_param).resolve_legend(color='independent').properties(
        title=alt.TitleParams(
            text='Clustering Results  ·  PCA 2D Projection',
            subtitle=[
                f'K-Means with era-specific optimal k  ·  {k_values_text}  ·  Use radio buttons to toggle time periods',
                'Click a point to isolate its cluster  ·  Diamonds = cluster centroids  ·  ★ gig-work & mlm-work tickers highlighted',
            ],
            fontSize=22, subtitleFontSize=13, subtitleColor='#888888', color='#1A1A1A', dy=-6
        ),
        width=680, height=460
    ).configure_view(strokeWidth=0, fill='#FAFAFA').configure_axis(
        labelFontSize=12, titleFontSize=14, titleColor='#555555',
        labelColor='#777777', domainColor='#CCCCCC', tickColor='#CCCCCC', gridOpacity=0.6
    ).configure_legend(
        labelFontSize=13, titleFontSize=14, titleColor='#333333', labelColor='#444444',
        orient='right', padding=10, cornerRadius=6, fillColor='#FFFFFF', strokeColor='#E8E8E8'
    )


pca_vis   = PCA(n_components=2, random_state=42)
X_pca_vis = pca_vis.fit_transform(reduced_X_scaled_minus_noise)

combined_pca_df  = build_era_pca_df(era_datasets_raw, era_optimal_k, pca_vis, X_pca_vis, feature_cols)
centroids_vis_df = build_centroids_df(combined_pca_df)
highlight_rows   = build_highlight_df(combined_pca_df, selected_features_minus_noise)

plot_clustering_vis(combined_pca_df, centroids_vis_df, highlight_rows, era_optimal_k, pca_vis, feature_cols)


alt.LayerChart(...)

In [112]:
# Cluster membership for gig-work & mlm-work companies per era
# combine HIGHLIGHT CATS AND NON-HIGHLIGHT CATS dictionaries strings into one set for filtering
NONHIGHLIGHT_CATS = {'sp500'}
combined_cats = HIGHLIGHT_CATS.union(NONHIGHLIGHT_CATS)

highlight_membership = (
    combined_pca_df[combined_pca_df['Category'].isin(combined_cats)]
    [['Era', 'Ticker', 'Company', 'Category', 'Cluster']]
    .sort_values(['Era', 'Category', 'Cluster', 'Ticker'])
    .reset_index(drop=True)
)

# Display one table per era
era_order = ['All Periods', 'Pre-COVID (2016-2019)', 'COVID (2020-2021)', 'Post-COVID (2022-2025)']

for era in era_order:
    era_df = highlight_membership[highlight_membership['Era'] == era][['Ticker', 'Company', 'Category', 'Cluster']]
    k = era_optimal_k[era]
    print(f"\n{'='*60}")
    print(f"  {era}  (k={k})")
    print(f"{'='*60}")
    print(era_df.to_string(index=False))
    print(f'Total in each cluster for {era}:')
    print(era_df['Cluster'].value_counts())


  All Periods  (k=8)
Ticker                                Company Category Cluster
  CART                              Instacart gig-work       1
  UBER                                   Uber gig-work       1
  DASH                               Doordash gig-work       2
  ETSY                                   Etsy gig-work       2
  FVRR                                 Fiverr gig-work       2
  LYFT                                   Lyft gig-work       2
  UDMY                                  Udemy gig-work       2
  UPWK                                 Upwork gig-work       2
  SHOP                                Shopify gig-work       3
   HLF                              Herbalife mlm-work       2
   PRI                              Primerica mlm-work       2
  USNA                                  USANA mlm-work       2
   NUS                                Nu Skin mlm-work       7
  CHRW                          C.H. Robinson    sp500       0
  CRWD                           